In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv
/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv


In [2]:
!python -m spacy download en_core_web_lg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler

import spacy


In [4]:
ARTICLE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv"
SENTENCE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv"

print(os.path.exists(ARTICLE_PATH), ARTICLE_PATH)
print(os.path.exists(SENTENCE_PATH), SENTENCE_PATH)


True /kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv
True /kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv


In [5]:
article_df = pd.read_csv(ARTICLE_PATH)
sentence_df = pd.read_csv(SENTENCE_PATH)

print("Article shape:", article_df.shape)
print(article_df.head())
print(article_df.columns.tolist())

print("Sentence shape:", sentence_df.shape)
print(sentence_df.head())
print(sentence_df.columns.tolist())


Article shape: (1018, 3)
   Unnamed: 0                                            article  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  There are a variety of emerging applications f...      0
2           2  As each new means of communication and social ...      0
3           3  These suggestions include:, Learn about the pu...      0
4           4  In recent years there has been growing concern...      0
['Unnamed: 0', 'article', 'class']
Sentence shape: (7344, 3)
   Unnamed: 0                                           sentence  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  In terms of linguistics, a program must be abl...      0
2           2  Of course each language has its own forms of a...      0
3           3  Programs can use several strategies for dealin...      0
4           4  As formidable as the task of extracting the co...      0
['Unnamed: 0', 'sentence', 'class']


In [6]:
def detect_text_label_columns(df):
    cols = [c.lower() for c in df.columns]
    text_candidates = ["text", "content", "article", "sentence", "body", "response", "prompt"]
    label_candidates = ["label", "class", "target", "y", "generated"]

    text_col = None
    label_col = None

    for c in df.columns:
        if c.lower() in text_candidates:
            text_col = c
            break

    for c in df.columns:
        if c.lower() in label_candidates:
            label_col = c
            break

    if text_col is None:
        for c in df.columns:
            if df[c].dtype == "object":
                text_col = c
                break

    if label_col is None:
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 5:
                label_col = c
                break

    return text_col, label_col

article_text_col, article_label_col = detect_text_label_columns(article_df)
sentence_text_col, sentence_label_col = detect_text_label_columns(sentence_df)

print("Article text:", article_text_col, "label:", article_label_col)
print("Sentence text:", sentence_text_col, "label:", sentence_label_col)


Article text: article label: class
Sentence text: sentence label: class


In [7]:
# This is highly minimized
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\u200b-\u200f\uFEFF]", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

article_df[article_text_col] = article_df[article_text_col].apply(clean_text)
sentence_df[sentence_text_col] = sentence_df[sentence_text_col].apply(clean_text)

print(article_df[[article_text_col, article_label_col]].head())
print(sentence_df[[sentence_text_col, sentence_label_col]].head())


                                             article  class
0  NLP is a multidisciplinary field that draws fr...      0
1  There are a variety of emerging applications f...      0
2  As each new means of communication and social ...      0
3  These suggestions include:, Learn about the pu...      0
4  In recent years there has been growing concern...      0
                                            sentence  class
0  NLP is a multidisciplinary field that draws fr...      0
1  In terms of linguistics, a program must be abl...      0
2  Of course each language has its own forms of a...      0
3  Programs can use several strategies for dealin...      0
4  As formidable as the task of extracting the co...      0


In [8]:
nlp = spacy.load("en_core_web_lg")


In [9]:
def get_spacy_vector(text):
    doc = nlp(text)
    vec = doc.vector
    if vec is None or vec.shape[0] == 0:
        return np.zeros(nlp.vocab.vectors_length, dtype=np.float32)
    return vec.astype(np.float32)

X_article = np.vstack(article_df[article_text_col].map(get_spacy_vector).values)
y_article = article_df[article_label_col].values

X_sentence = np.vstack(sentence_df[sentence_text_col].map(get_spacy_vector).values)
y_sentence = sentence_df[sentence_label_col].values

print(X_article.shape, X_sentence.shape)


(1018, 300) (7344, 300)


In [10]:
# train test split

def prepare_labels(y):
    if y.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y)
        return y, le
    return y, None

y_article, article_le = prepare_labels(y_article)
y_sentence, sentence_le = prepare_labels(y_sentence)

X_article_train, X_article_test, y_article_train, y_article_test = train_test_split(
    X_article, y_article, test_size=0.2, random_state=42, shuffle=True, stratify=y_article
)

X_sentence_train, X_sentence_test, y_sentence_train, y_sentence_test = train_test_split(
    X_sentence, y_sentence, test_size=0.2, random_state=42, shuffle=True, stratify=y_sentence
)

print(X_article_train.shape, X_article_test.shape)
print(X_sentence_train.shape, X_sentence_test.shape)


(814, 300) (204, 300)
(5875, 300) (1469, 300)


In [11]:
models = {
    "MultinomialNB": Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", MultinomialNB(alpha=1.0, fit_prior=True))
    ]),
    
    "SVM_poly": Pipeline([
        ("scaler", MaxAbsScaler()),
        ("clf", SVC(kernel="poly", degree=6, probability=True))
    ]),
    
    "RandomForest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
    ]),
    
    "KNN": Pipeline([
        ("scaler", MaxAbsScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, metric="euclidean"))
    ]),
}

In [12]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name="model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    y_prob = None
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        y_prob = scores

    acc = accuracy_score(y_test, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"=== {name} ===")
    print("Accuracy:", acc)
    print("Precision:", p)
    print("Recall:", r)
    print("F1:", f1)
    print("Confusion matrix:\n", cm)
    print(classification_report(y_test, y_pred, zero_division=0))

    if y_prob is not None and len(np.unique(y_test)) == 2:
        try:
            auc = roc_auc_score(y_test, y_prob)
            print("ROC-AUC:", auc)
        except Exception as e:
            print("ROC-AUC failed:", e)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
        "cm": cm
    }


In [13]:
sentence_results = {}

for name, model in models.items():
    sentence_results[name] = evaluate_model(
        model,
        X_sentence_train,
        y_sentence_train,
        X_sentence_test,
        y_sentence_test,
        name=f"Sentence - {name}"
    )


=== Sentence - MultinomialNB ===
Accuracy: 0.6569094622191968
Precision: 0.6352087114337568
Recall: 0.8728179551122195
F1: 0.7352941176470589
Confusion matrix:
 [[265 402]
 [102 700]]
              precision    recall  f1-score   support

           0       0.72      0.40      0.51       667
           1       0.64      0.87      0.74       802

    accuracy                           0.66      1469
   macro avg       0.68      0.64      0.62      1469
weighted avg       0.67      0.66      0.63      1469

ROC-AUC: 0.7693977200925721
=== Sentence - SVM_poly ===
Accuracy: 0.8012253233492171
Precision: 0.8541666666666666
Recall: 0.7668329177057357
F1: 0.8081471747700394
Confusion matrix:
 [[562 105]
 [187 615]]
              precision    recall  f1-score   support

           0       0.75      0.84      0.79       667
           1       0.85      0.77      0.81       802

    accuracy                           0.80      1469
   macro avg       0.80      0.80      0.80      1469
weighted a